In [1]:
%cd /Users/vmasrani/dev/sophia-opensource-projects/pmap

/Users/vmasrani/dev/sophia-opensource-projects/pmap


# pmap Notebook Test

Tests that `pmap` auto-detects notebook environment and uses the tqdm backend.
Also tests explicit backend override with `backend='tqdm'`.

In [2]:
import time
from loguru import logger
import rich
from pmap import pmap, is_notebook

ITEMS = range(5)
EXPECTED = [0, 2, 4, 6, 8]

def worker(i: int) -> int:
    """Worker with all output methods."""
    print(f"print: item {i}")
    rich.print(f"[bold cyan]rich: item {i}[/bold cyan]")
    logger.info(f"loguru: item {i}")
    time.sleep(0.2)
    return i * 2

def worker_that_fails(i: int) -> int:
    if i % 2 == 1:
        raise ValueError(f"Item {i} is odd")
    return i * 2

print(f"is_notebook: {is_notebook()}")

is_notebook: True


## Test 1: Auto backend (should use tqdm in notebook)

In [3]:
results = pmap(worker, ITEMS, n_jobs=2)
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

Processing:   0%|          | 0/5 [00:00<?, ?it/s]

13:57:44 | INFO     | print: item 1
13:57:44 | INFO     | print: item 0
13:57:44 | INFO     | rich: item 1
13:57:44 | INFO     | loguru: item 1
13:57:44 | INFO     | rich: item 0
13:57:44 | INFO     | loguru: item 0
13:57:44 | INFO     | print: item 2
13:57:44 | INFO     | rich: item 2
13:57:44 | INFO     | print: item 3
13:57:44 | INFO     | loguru: item 2
13:57:44 | INFO     | rich: item 3
13:57:44 | INFO     | loguru: item 3
13:57:44 | INFO     | print: item 4
13:57:44 | INFO     | rich: item 4
13:57:44 | INFO     | loguru: item 4
Results: [0, 2, 4, 6, 8] — PASSED


## Test 2: Processes + Job Bars (falls back to simple bar in notebook)

In [4]:
results = pmap(worker, ITEMS, n_jobs=2, show_job_bars=True)
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

Processing:   0%|          | 0/5 [00:00<?, ?it/s]

13:57:47 | INFO     | print: item 0
13:57:47 | INFO     | rich: item 0
13:57:47 | INFO     | loguru: item 0
13:57:47 | INFO     | print: item 1
13:57:47 | INFO     | rich: item 1
13:57:47 | INFO     | loguru: item 1
13:57:48 | INFO     | print: item 2
13:57:48 | INFO     | rich: item 2
13:57:48 | INFO     | loguru: item 2
13:57:48 | INFO     | print: item 3
13:57:48 | INFO     | rich: item 3
13:57:48 | INFO     | loguru: item 3
13:57:48 | INFO     | print: item 4
13:57:48 | INFO     | rich: item 4
13:57:48 | INFO     | loguru: item 4
Results: [0, 2, 4, 6, 8] — PASSED


## Test 3: Threads + Simple Bar

In [5]:
results = pmap(worker, ITEMS, n_jobs=2, prefer='threads')
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

Processing:   0%|          | 0/5 [00:00<?, ?it/s]

print: item 0print: item 1



rich: item 0

13:58:15 | INFO     | loguru: item 0


rich: item 1

...

13:58:16 | INFO     | loguru: item 2


rich: item 3

13:58:16 | INFO     | loguru: item 3
print: item 4


rich: item 4

13:58:16 | INFO     | loguru: item 4
Results: [0, 2, 4, 6, 8] — PASSED


## Test 4: Threads + Job Bars

In [6]:
results = pmap(worker, ITEMS, n_jobs=2, prefer='threads', show_job_bars=True)
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

Processing:   0%|          | 0/5 [00:00<?, ?it/s]

print: item 0print: item 1


rich: item 1


13:58:20 | INFO     | loguru: item 1


rich: item 0

...

13:58:20 | INFO     | loguru: item 2
print: item 3


rich: item 3

13:58:20 | INFO     | loguru: item 3
print: item 4


rich: item 4

13:58:20 | INFO     | loguru: item 4
Results: [0, 2, 4, 6, 8] — PASSED


## Test 5: Sequential (n_jobs=1)

In [7]:
results = pmap(worker, ITEMS, n_jobs=1)
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

Processing:   0%|          | 0/5 [00:00<?, ?it/s]

print: item 0


rich: item 0

12:18:35 | INFO     | loguru: item 0


Processing:  20%|██        | 1/5 [00:00<00:00,  4.81it/s]

...

print: item 4


rich: item 4

12:18:36 | INFO     | loguru: item 4


Processing: 100%|██████████| 5/5 [00:01<00:00,  4.84it/s]

Results: [0, 2, 4, 6, 8] — PASSED


## Test 6: Progress Disabled

In [7]:
results = pmap(worker, ITEMS, n_jobs=2, disable_tqdm=True)
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

13:58:28 | INFO     | print: item 0
13:58:28 | INFO     | rich: item 0
13:58:28 | INFO     | loguru: item 0
13:58:28 | INFO     | print: item 1
13:58:28 | INFO     | rich: item 1
13:58:28 | INFO     | loguru: item 1
13:58:29 | INFO     | print: item 2
13:58:29 | INFO     | rich: item 2
13:58:29 | INFO     | loguru: item 2
13:58:29 | INFO     | print: item 3
13:58:29 | INFO     | rich: item 3
13:58:29 | INFO     | loguru: item 3
13:58:29 | INFO     | print: item 4
13:58:29 | INFO     | rich: item 4
13:58:29 | INFO     | loguru: item 4
Results: [0, 2, 4, 6, 8] — PASSED


## Test 7: safe_mode

In [8]:
results = pmap(worker_that_fails, range(4), n_jobs=2, safe_mode=True)
assert results[0] == 0
assert isinstance(results[1], dict) and results[1]['error_type'] == 'ValueError'
assert results[2] == 4
assert isinstance(results[3], dict) and results[3]['error_type'] == 'ValueError'
print(f"Results: {results} — PASSED")

Processing:   0%|          | 0/4 [00:00<?, ?it/s]

Results: [0, {'error': 'Item 1 is odd', 'error_type': 'ValueError', 'args': (1,), 'kwargs': {}}, 4, {'error': 'Item 3 is odd', 'error_type': 'ValueError', 'args': (3,), 'kwargs': {}}] — PASSED


## Test 8: Explicit backend='tqdm' override

In [9]:
results = pmap(worker, ITEMS, n_jobs=2, backend='tqdm')
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

Processing:   0%|          | 0/5 [00:00<?, ?it/s]

13:58:34 | INFO     | print: item 0
13:58:34 | INFO     | rich: item 0
13:58:34 | INFO     | loguru: item 0
13:58:34 | INFO     | print: item 1
13:58:34 | INFO     | rich: item 1
13:58:34 | INFO     | loguru: item 1
13:58:34 | INFO     | print: item 2
13:58:34 | INFO     | rich: item 2
13:58:34 | INFO     | loguru: item 2
13:58:34 | INFO     | print: item 3
13:58:34 | INFO     | rich: item 3
13:58:34 | INFO     | loguru: item 3
13:58:35 | INFO     | print: item 4
13:58:35 | INFO     | rich: item 4
13:58:35 | INFO     | loguru: item 4
Results: [0, 2, 4, 6, 8] — PASSED


## All Tests Complete

In [ ]:
print("=" * 60)
print("ALL NOTEBOOK TESTS PASSED!")
print("=" * 60)